# ImageCLEF 2026 – Deepfake Detection
**Runtime > Change runtime type > T4 GPU 선택 후 실행**

## 순서
1. Google Drive 마운트
2. 패키지 설치
3. 데이터 압축 해제
4. 이미지 딥페이크 탐지
5. 오디오 딥페이크 탐지
6. CSV 저장

## 1. Google Drive 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. 패키지 설치
> 약 2~3분 소요

In [ ]:
!pip install -q transformers timm torchaudio librosa soundfile pillow pandas tqdm
!pip install -q accelerate

## 3. 데이터 압축 해제
> **ZIP 파일 경로를 본인 Drive 경로에 맞게 수정하세요**

In [ ]:
import zipfile, os

# ← 여기 경로를 본인 Drive에 업로드한 zip 경로로 변경
ZIP_PATH = '/content/drive/MyDrive/ImageCLEF2026-DeepFakeDetection-Tes.zip'
EXTRACT_PATH = '/content/data'

if not os.path.exists(EXTRACT_PATH):
    print('압축 해제 중...')
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall(EXTRACT_PATH)
    print('완료!')
else:
    print('이미 압축 해제됨')

# 경로 확인
IMAGE_DIR  = '/content/data/Data/Images_Detection'
AUDIO_DIR  = '/content/data/Data/Audio_Detection'
IMG_CSV    = '/content/data/Data/Images_Detection_submission.csv'
AUD_CSV    = '/content/data/Data/Audio_Detection_submission.csv'

import glob
print(f'이미지 파일 수: {len(glob.glob(IMAGE_DIR+"/*.png"))}')
print(f'오디오 파일 수: {len(glob.glob(AUDIO_DIR+"/*.wav"))}')

## 4. 이미지 딥페이크 탐지
모델: `prithivMLmods/Deepfake-vs-Real-Image-Detection` (ViT 기반)
> 약 20~30분 소요 (T4 GPU 기준)

In [ ]:
import torch
import pandas as pd
from PIL import Image
from transformers import pipeline
from tqdm.auto import tqdm

print('GPU 사용 여부:', torch.cuda.is_available())
device = 0 if torch.cuda.is_available() else -1

print('이미지 탐지 모델 로드 중...')
img_pipe = pipeline(
    'image-classification',
    model='prithivMLmods/Deepfake-vs-Real-Image-Detection',
    device=device
)
print('모델 로드 완료!')

In [ ]:
import os

img_df = pd.read_csv(IMG_CSV)
print(f'총 이미지: {len(img_df)}개')

predictions = []
BATCH_SIZE = 32

filenames = img_df['full_secret_name'].tolist()

for i in tqdm(range(0, len(filenames), BATCH_SIZE), desc='이미지 추론'):
    batch_names = filenames[i:i+BATCH_SIZE]
    batch_images = []
    valid_indices = []

    for j, fname in enumerate(batch_names):
        fpath = os.path.join(IMAGE_DIR, fname)
        try:
            img = Image.open(fpath).convert('RGB')
            batch_images.append(img)
            valid_indices.append(j)
        except Exception as e:
            print(f'오류: {fname} - {e}')
            batch_images.append(None)

    results = img_pipe([img for img in batch_images if img is not None])

    res_iter = iter(results)
    for j, fname in enumerate(batch_names):
        if j in valid_indices:
            r = next(res_iter)
            label = r['label'].lower()
            # 'deepfake' or 'fake' → 1, 'real' → 0
            if 'fake' in label or 'artificial' in label or 'ai' in label:
                predictions.append(1)
            else:
                predictions.append(0)
        else:
            predictions.append(0)  # 오류난 파일은 Real로 처리

img_df['prediction'] = predictions
print(f'\n예측 완료!')
print(img_df['prediction'].value_counts())

In [ ]:
# 이미지 결과 저장 (Drive에도 백업)
img_out = '/content/Images_Detection_submission.csv'
img_df.to_csv(img_out, index=False)

# Drive 백업
import shutil
shutil.copy(img_out, '/content/drive/MyDrive/Images_Detection_submission.csv')
print('이미지 CSV 저장 완료!')
img_df.head()

## 5. 오디오 딥페이크 탐지
모델: AASIST (Anti-Spoofing using Spectro-Temporal graph)
> 약 30~40분 소요 (T4 GPU 기준)

In [ ]:
# AASIST 공식 레포 클론 + 사전학습 가중치 다운로드
!git clone --quiet https://github.com/clovaai/aasist.git /content/aasist
%cd /content/aasist
!pip install -q -r requirements.txt

# 사전학습 가중치 다운로드 (AASIST-L)
import os
os.makedirs('/content/aasist/models/weights', exist_ok=True)
!gdown --id 1-4-MViVNRwJKqsMOCOhPJuVZlasHkYAl -O /content/aasist/models/weights/AASIST-L.pth 2>/dev/null || \
 wget -q -O /content/aasist/models/weights/AASIST-L.pth \
   'https://github.com/clovaai/aasist/releases/download/v1.0.0/AASIST-L.pth'
print('AASIST 설정 완료!')

In [ ]:
import sys
sys.path.insert(0, '/content/aasist')

import torch
import torchaudio
import json
import numpy as np
from tqdm.auto import tqdm

from models.AASIST import Model as AASIST

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# AASIST-L 설정
with open('/content/aasist/config/AASIST-L.conf', 'r') as f:
    config = json.load(f)

model = AASIST(config['model_config']).to(device)
weights = torch.load('/content/aasist/models/weights/AASIST-L.pth', map_location=device)
model.load_state_dict(weights)
model.eval()
print('AASIST 모델 로드 완료!')

In [ ]:
TARGET_SR = 16000
MAX_LEN   = 64600  # AASIST 기본 입력 길이 (~4초)

def load_audio(fpath):
    wav, sr = torchaudio.load(fpath)
    if sr != TARGET_SR:
        wav = torchaudio.functional.resample(wav, sr, TARGET_SR)
    wav = wav.mean(dim=0)  # stereo → mono
    # 길이 맞추기
    if wav.shape[0] < MAX_LEN:
        wav = torch.nn.functional.pad(wav, (0, MAX_LEN - wav.shape[0]))
    else:
        wav = wav[:MAX_LEN]
    return wav

aud_df = pd.read_csv(AUD_CSV)
print(f'총 오디오: {len(aud_df)}개')

aud_predictions = []
BATCH_SIZE = 16

filenames = aud_df['full_secret_name'].tolist()

with torch.no_grad():
    for i in tqdm(range(0, len(filenames), BATCH_SIZE), desc='오디오 추론'):
        batch_names = filenames[i:i+BATCH_SIZE]
        batch_wavs  = []
        valid_mask  = []

        for fname in batch_names:
            fpath = os.path.join(AUDIO_DIR, fname)
            try:
                wav = load_audio(fpath)
                batch_wavs.append(wav)
                valid_mask.append(True)
            except Exception as e:
                print(f'오류: {fname} - {e}')
                batch_wavs.append(torch.zeros(MAX_LEN))
                valid_mask.append(False)

        x = torch.stack(batch_wavs).to(device)  # (B, T)
        _, out = model(x)  # out shape: (B, 2) — [real_score, fake_score]
        probs = torch.softmax(out, dim=1)[:, 1]  # fake 확률

        for k, p in enumerate(probs.cpu().numpy()):
            if valid_mask[k]:
                aud_predictions.append(1 if p >= 0.5 else 0)
            else:
                aud_predictions.append(0)

aud_df['prediction'] = aud_predictions
print(f'\n예측 완료!')
print(aud_df['prediction'].value_counts())

In [ ]:
# 오디오 결과 저장
aud_out = '/content/Audio_Detection_submission.csv'
aud_df.to_csv(aud_out, index=False)

import shutil
shutil.copy(aud_out, '/content/drive/MyDrive/Audio_Detection_submission.csv')
print('오디오 CSV 저장 완료!')
aud_df.head()

## 6. 최종 확인 & 제출 파일 준비

In [ ]:
import zipfile

# 제출용 zip 생성
submit_zip = '/content/submission.zip'
with zipfile.ZipFile(submit_zip, 'w') as z:
    z.write('/content/Images_Detection_submission.csv', 'Images_Detection_submission.csv')
    z.write('/content/Audio_Detection_submission.csv',  'Audio_Detection_submission.csv')

shutil.copy(submit_zip, '/content/drive/MyDrive/submission.zip')
print('제출 파일 생성 완료: submission.zip')

# 최종 확인
img_df2 = pd.read_csv('/content/Images_Detection_submission.csv')
aud_df2 = pd.read_csv('/content/Audio_Detection_submission.csv')

print(f'\n[이미지] 총 {len(img_df2)}개 | 빈값: {img_df2["prediction"].isna().sum()}')
print(img_df2['prediction'].value_counts())
print(f'\n[오디오] 총 {len(aud_df2)}개 | 빈값: {aud_df2["prediction"].isna().sum()}')
print(aud_df2['prediction'].value_counts())

In [ ]:
# (선택) Colab에서 직접 다운로드
from google.colab import files
files.download('/content/submission.zip')